In [2]:
from pathlib import Path
import getpass
from langchain_community import document_loaders
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_openrouter import ChatOpenRouter
import chromadb
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
import os
import dotenv
from dotenv import load_dotenv
load_dotenv()
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.output_parsers import StrOutputParser

C:\Users\vansh_hxqeh4o\AppData\Local\Temp\ipykernel_7336\361796743.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community import document_loaders


In [3]:
path=r"D:\GEN AI BASICS\RAG\RETRIEVER-FULL\Data\llama2-research-paper.pdf"
loader = PyPDFLoader(path)
pages=loader.load()
print(f"Total Pages: {len(pages)}")

Total Pages: 77


In [4]:
def identify_section(paper_page):
    """
    Identify the major section of the Llama 2 paper
    using its printed PDF page number.
    """

    if 1 <= paper_page <= 2:
        return "front_matter"

    if 3 <= paper_page <= 4:
        return "introduction"

    if 5 <= paper_page <= 7:
        return "pretraining"

    if 8 <= paper_page <= 19:
        return "fine_tuning"

    if 20 <= paper_page <= 31:
        return "safety"

    if 32 <= paper_page <= 35:
        return "discussion"

    if paper_page == 36:
        return "conclusion"

    if 37 <= paper_page <= 45:
        return "references"

    if 46 <= paper_page <= 77:
        return "appendix"

    return "unknown"


In [5]:
for pagedoc in pages:
    page_index=pagedoc.metadata.get("page", 0)
    paper_page=page_index+1
    pagedoc.metadata.update({
        "paper": "Llama 2",
        "organization": "Meta",
        "year": 2023,
        "document_type": "research_paper",
        "paper_page": paper_page,
        "section": identify_section(paper_page),
        "access_level": "public",
    }
    )

In [6]:
text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
   add_start_index=True,
)
chunked_docs=text_splitter.split_documents(pages)
print(f"Total pages: {len(pages)}")
print(f"Total Chunks: {len(chunked_docs)}")

Total pages: 77
Total Chunks: 317


In [7]:
for i,chunks in enumerate(chunked_docs):
    paper_page=chunks.metadata.get("paper_page", "unknown")
    chunks.metadata["chunk_id"]=f"llama2-page-{paper_page}-chunk-{i}"

In [8]:
print(f"Chunk 0 metadata: {chunked_docs[0].metadata}")

Chunk 0 metadata: {'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'D:\\GEN AI BASICS\\RAG\\RETRIEVER-FULL\\Data\\llama2-research-paper.pdf', 'total_pages': 77, 'page': 0, 'page_label': '1', 'paper': 'Llama 2', 'organization': 'Meta', 'year': 2023, 'document_type': 'research_paper', 'paper_page': 1, 'section': 'front_matter', 'access_level': 'public', 'start_index': 0, 'chunk_id': 'llama2-page-1-chunk-0'}


In [9]:
embeddings_hf = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
dimensions=embeddings_hf.embed_query("Hello")
print(f"Embedding dimensions: {len(dimensions)}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding dimensions: 384


In [10]:
vector_store=Chroma(
    collection_name="llama2-research-paper",
    embedding_function=embeddings_hf,
    persist_directory=r"D:\GEN AI BASICS\RAG\RETRIEVER-FULL\Data\chroma",
    collection_metadata={
        "hnsw:space": "cosine",
    }
)

In [11]:
documents=vector_store.add_documents(chunked_docs)
print(f"Total documents added to vector store: {len(documents)}")

Total documents added to vector store: 317


In [12]:
similarity_retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})
query="How does LLaMA 2 ensure safety in its models?"
response=similarity_retriever.invoke(query)

In [38]:
def display_documents(documents, title="", max_documents=None):

    print("=" * 90)
    print(title)
    print("=" * 90)

    if max_documents is not None:
        documents = documents[:max_documents]

    for rank, document in enumerate(documents, start=1):
        metadata = document.metadata

        print(f"RANK: {rank}")
        print(f"PAPER PAGE: {metadata.get('paper_page')}")
        print(f"SECTION: {metadata.get('section')}")
        print(f"CHUNK ID: {metadata.get('chunk_id')}")
        print(f"SOURCE: {metadata.get('source')}")
        print("-" * 90)
        print(document.page_content[:500])
        print()

In [14]:
similarity_retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})
query = "How does LLaMA 2 ensure safety in its models?"
response = similarity_retriever.invoke(query)
display_documents(response)

RANK: 1
PAPER PAGE: 77
SECTION: appendix
CHUNK ID: llama2-page-77-chunk-316
SOURCE: D:\GEN AI BASICS\RAG\RETRIEVER-FULL\Data\llama2-research-paper.pdf
------------------------------------------------------------------------------------------
Ethical Considerations and Limitations(Section 5.2)
Llama 2is a new technology that carries risks with use. Testing conducted to date has been in
English, and has not covered, nor could it cover all scenarios. For these reasons, as with all LLMs,
Llama 2’s potential outputs cannot be predicted in advance, and the model may in some instances
produce inaccurate or objectionable responses to user prompts. Therefore, before deploying any
applications ofLlama 2, developers should perform safety testi

RANK: 2
PAPER PAGE: 77
SECTION: appendix
CHUNK ID: llama2-page-77-chunk-316
SOURCE: D:\GEN AI BASICS\RAG\RETRIEVER-FULL\Data\llama2-research-paper.pdf
------------------------------------------------------------------------------------------
Ethical Consid

In [15]:
mmr_retriever=vector_store.as_retriever(
    search_type="mmr", search_kwargs={"k": 4, "fetch_k": 20, "lambda_mult": 0.5}
)
query="How does LLaMA 2 ensure safety in its models?"
mmr_response=mmr_retriever.invoke(query)
display_documents(mmr_response)

RANK: 1
PAPER PAGE: 77
SECTION: appendix
CHUNK ID: llama2-page-77-chunk-316
SOURCE: D:\GEN AI BASICS\RAG\RETRIEVER-FULL\Data\llama2-research-paper.pdf
------------------------------------------------------------------------------------------
Ethical Considerations and Limitations(Section 5.2)
Llama 2is a new technology that carries risks with use. Testing conducted to date has been in
English, and has not covered, nor could it cover all scenarios. For these reasons, as with all LLMs,
Llama 2’s potential outputs cannot be predicted in advance, and the model may in some instances
produce inaccurate or objectionable responses to user prompts. Therefore, before deploying any
applications ofLlama 2, developers should perform safety testi

RANK: 2
PAPER PAGE: 4
SECTION: introduction
CHUNK ID: llama2-page-4-chunk-11
SOURCE: D:\GEN AI BASICS\RAG\RETRIEVER-FULL\Data\llama2-research-paper.pdf
------------------------------------------------------------------------------------------
safety testin

In [16]:
threshold_retriever = vector_store. as_retriever(
search_type="similarity_score_threshold",
search_kwargs={
"k": 10,
"score_threshold": 0.50,
}
)
query = "What safety techniques were used for Llama 2-Chat?"
threshold_response = threshold_retriever.invoke(query)
display_documents (threshold_response)

RANK: 1
PAPER PAGE: 4
SECTION: introduction
CHUNK ID: llama2-page-4-chunk-11
SOURCE: D:\GEN AI BASICS\RAG\RETRIEVER-FULL\Data\llama2-research-paper.pdf
------------------------------------------------------------------------------------------
safety testing and tuning tailored to their specific applications of the model. We provide a responsible use
guide¶ and code examples‖ to facilitate the safe deployment ofLlama 2 and Llama 2-Chat. More details of
our responsible release strategy can be found in Section 5.3.
The remainder of this paper describes our pretraining methodology (Section 2), fine-tuning methodology
(Section 3), approach to model safety (Section 4), key observations and insights (Section 5), relevant related
work (Secti

RANK: 2
PAPER PAGE: 4
SECTION: introduction
CHUNK ID: llama2-page-4-chunk-11
SOURCE: D:\GEN AI BASICS\RAG\RETRIEVER-FULL\Data\llama2-research-paper.pdf
------------------------------------------------------------------------------------------
safety testi

## Pre-filtering

In [17]:
fine_tunning_retriever = vector_store.as_retriever(
    search_type="similarity", 
    search_kwargs={"k": 4,
                   "filter":{
                       "section": "fine_tuning"
                   }
}
)
query="how was llama 2-chat alligned with human preferences?"
fine_tuning_response = fine_tunning_retriever.invoke(query)
display_documents(fine_tuning_response)

RANK: 1
PAPER PAGE: 18
SECTION: fine_tuning
CHUNK ID: llama2-page-18-chunk-70
SOURCE: D:\GEN AI BASICS\RAG\RETRIEVER-FULL\Data\llama2-research-paper.pdf
------------------------------------------------------------------------------------------
results are presented in Section 4.4.
Results. As shown in Figure 12,Llama 2-Chat models outperform open-source models by a significant
margin on both single turn and multi-turn prompts. Particularly,Llama 2-Chat 7B model outperforms
MPT-7B-chat on 60% of the prompts.Llama 2-Chat 34B has an overall win rate of more than 75% against
equivalently sized Vicuna-33B and Falcon 40B models.
18

RANK: 2
PAPER PAGE: 18
SECTION: fine_tuning
CHUNK ID: llama2-page-18-chunk-70
SOURCE: D:\GEN AI BASICS\RAG\RETRIEVER-FULL\Data\llama2-research-paper.pdf
------------------------------------------------------------------------------------------
results are presented in Section 4.4.
Results. As shown in Figure 12,Llama 2-Chat models outperform open-source models by

In [18]:
pre_filter = {
    "$and": [
        {"section": {"$eq": "fine_tuning"}},
        {"year": {"$eq": 2023}}
    ]
}
prefiltered_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4, "filter": pre_filter}
)

In [19]:
query="How was the reward model trained?"
prefiltered_response = prefiltered_retriever.invoke(query)
display_documents(prefiltered_response)

RANK: 1
PAPER PAGE: 17
SECTION: fine_tuning
CHUNK ID: llama2-page-17-chunk-66
SOURCE: D:\GEN AI BASICS\RAG\RETRIEVER-FULL\Data\llama2-research-paper.pdf
------------------------------------------------------------------------------------------
increase iteration speed. We later validated major model versions with human evaluations.
How Far Can Model-Based Evaluation Go?To measure the robustness of our reward model, we collected
a test set of prompts for both helpfulness and safety, and asked three annotators to judge the quality of the
answers based on a 7-point Likert scale (the higher the better). We observe that our reward models overall
are well calibrated with our human preference annotations, as illustrated in Figure 29 in the a

RANK: 2
PAPER PAGE: 17
SECTION: fine_tuning
CHUNK ID: llama2-page-17-chunk-66
SOURCE: D:\GEN AI BASICS\RAG\RETRIEVER-FULL\Data\llama2-research-paper.pdf
------------------------------------------------------------------------------------------
increase i

## Post-Filtering

In [22]:
unfiltered_retriever = vector_store.similarity_search_with_score(
    query,
    k=20
)

In [24]:
post_filtered_docs = [
    document
    for document, score in unfiltered_retriever
    if document.metadata.get("section") == "fine_tuning"
    and document.metadata.get("year") == 2023
]
display_documents(post_filtered_docs)

RANK: 1
PAPER PAGE: 17
SECTION: fine_tuning
CHUNK ID: llama2-page-17-chunk-66
SOURCE: D:\GEN AI BASICS\RAG\RETRIEVER-FULL\Data\llama2-research-paper.pdf
------------------------------------------------------------------------------------------
increase iteration speed. We later validated major model versions with human evaluations.
How Far Can Model-Based Evaluation Go?To measure the robustness of our reward model, we collected
a test set of prompts for both helpfulness and safety, and asked three annotators to judge the quality of the
answers based on a 7-point Likert scale (the higher the better). We observe that our reward models overall
are well calibrated with our human preference annotations, as illustrated in Figure 29 in the a

RANK: 2
PAPER PAGE: 17
SECTION: fine_tuning
CHUNK ID: llama2-page-17-chunk-66
SOURCE: D:\GEN AI BASICS\RAG\RETRIEVER-FULL\Data\llama2-research-paper.pdf
------------------------------------------------------------------------------------------
increase i

In [25]:
print(f"Total documents retrieved after pre-filtering: {len(prefiltered_response)}")
print(f"Total documents retrieved after post-filtering: {len(post_filtered_docs)}")

Total documents retrieved after pre-filtering: 4
Total documents retrieved after post-filtering: 16


## RETRIEVER PART-2

In [27]:
import os 
from typing import List
from pydantic import BaseModel,Field 
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers.ensemble import EnsembleRetriever
from langchain_openrouter import ChatOpenRouter
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.prompts import PromptTemplate

from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

In [ ]:
def deduplicate_documents(documents):
    """remove duplicate documents based on their metadata and content"""
    unique_docs = []
    seen_keys=set()
    for document in documents:
        key=(
            document.metadata.get("chunk_id")
            or(
                document.metadata.get("source"),
                document.metadata.get("page"),
                document.page_content,
            )
        )
        if key not in seen_keys:
            seen_keys.add(key)
            unique_docs.append(document)
    return unique_docs


In [28]:
from langchain_community.retrievers import BM25Retriever
bm25_retriever=BM25Retriever.from_documents(chunked_docs)
#final number of result
bm25_retriever.k=4

In [32]:
sparse_query="Grouped-Query Attention GQA 70B"
sparse_docs=bm25_retriever.invoke(sparse_query)
display_documents(sparse_docs)

RANK: 1
PAPER PAGE: 6
SECTION: pretraining
CHUNK ID: llama2-page-6-chunk-15
SOURCE: D:\GEN AI BASICS\RAG\RETRIEVER-FULL\Data\llama2-research-paper.pdf
------------------------------------------------------------------------------------------
Training Data Params Context
Length
GQA Tokens LR
Llama 1 See Touvron et al.
(2023)
7B 2k ✗ 1.0T 3.0 × 10−4
13B 2k ✗ 1.0T 3.0 × 10−4
33B 2k ✗ 1.4T 1.5 × 10−4
65B 2k ✗ 1.4T 1.5 × 10−4
Llama 2 A new mix of publicly
available online data
7B 4k ✗ 2.0T 3.0 × 10−4
13B 4k ✗ 2.0T 3.0 × 10−4
34B 4k ✓ 2.0T 1.5 × 10−4
70B 4k ✓ 2.0T 1.5 × 10−4
Table 1:Llama 2 family of models.Token counts refer to pretraining data only. All models are trained with
a global batch-size of 4M tokens. Bigger models — 34B and 70

RANK: 2
PAPER PAGE: 48
SECTION: appendix
CHUNK ID: llama2-page-48-chunk-201
SOURCE: D:\GEN AI BASICS\RAG\RETRIEVER-FULL\Data\llama2-research-paper.pdf
------------------------------------------------------------------------------------------
BoolQ PIQA SIQ

In [34]:
dense_retriever=vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})
dense_query= ("How did Meta improve inference scalability "
              "for the largest Llama 2 models?")
dense_docs=dense_retriever.invoke(dense_query)
display_documents(dense_docs)


RANK: 1
PAPER PAGE: 8
SECTION: fine_tuning
CHUNK ID: llama2-page-8-chunk-23
SOURCE: D:\GEN AI BASICS\RAG\RETRIEVER-FULL\Data\llama2-research-paper.pdf
------------------------------------------------------------------------------------------
et al., 2020), Big Bench Hard (BBH) (3 shot) (Suzgun et al., 2022), and AGI Eval (3–5 shot) (Zhong
et al., 2023). For AGI Eval, we only evaluate on the English tasks and report the average.
As shown in Table 3,Llama 2 models outperformLlama 1 models. In particular,Llama 2 70B improves the
resultsonMMLUandBBHby ≈5and ≈8points, respectively, comparedtoLlama 1 65B. Llama 2 7Band30B
modelsoutperformMPTmodelsofthecorrespondingsizeonallcategoriesbesidescodebenchmarks. Forthe
Falcon models,Llama 2 7B a

RANK: 2
PAPER PAGE: 8
SECTION: fine_tuning
CHUNK ID: llama2-page-8-chunk-23
SOURCE: D:\GEN AI BASICS\RAG\RETRIEVER-FULL\Data\llama2-research-paper.pdf
------------------------------------------------------------------------------------------
et al., 2020),

In [39]:
#comparison of sparse and dense retrieval results
comparison_query = (
    "How did grouped-query attention improve "
    "Llama 2 inference scalability?"
)

sparse_results = bm25_retriever.invoke(comparison_query)
dense_results = dense_retriever.invoke(comparison_query)

display_documents(
    sparse_results,
    title="BM25 Results",
    max_documents=4,
)

display_documents(
    dense_results,
    title="Dense Vector Results",
    max_documents=4,
)

BM25 Results
RANK: 1
PAPER PAGE: 4
SECTION: introduction
CHUNK ID: llama2-page-4-chunk-10
SOURCE: D:\GEN AI BASICS\RAG\RETRIEVER-FULL\Data\llama2-research-paper.pdf
------------------------------------------------------------------------------------------
increased the size of the pretraining corpus by 40%, doubled the context length of the model, and
adopted grouped-query attention (Ainslie et al., 2023). We are releasing variants ofLlama 2 with
7B, 13B, and 70B parameters. We have also trained 34B variants, which we report on in this paper
but are not releasing.§
2. Llama 2-Chat, a fine-tuned version ofLlama 2 that is optimized for dialogue use cases. We release
variants of this model with 7B, 13B, and 70B parameters as well.
We believe that th

RANK: 2
PAPER PAGE: 6
SECTION: pretraining
CHUNK ID: llama2-page-6-chunk-15
SOURCE: D:\GEN AI BASICS\RAG\RETRIEVER-FULL\Data\llama2-research-paper.pdf
------------------------------------------------------------------------------------------


In [42]:
print("SPARSE RESULTS")
for rank, document in enumerate(sparse_results, start=1):
   print(
   rank,
   document.metadata.get("paper_page"),
   document.metadata.get("chunk_id"),
) 

print("\nDENSE RESULTS")
for rank, document in enumerate(dense_results, start=1):
   print(
   rank,
   document.metadata.get("paper_page"),
   document.metadata.get("chunk_id"),
)

SPARSE RESULTS
1 4 llama2-page-4-chunk-10
2 6 llama2-page-6-chunk-15
3 54 llama2-page-54-chunk-218
4 47 llama2-page-47-chunk-198

DENSE RESULTS
1 47 llama2-page-47-chunk-198
2 47 llama2-page-47-chunk-198
3 47 llama2-page-47-chunk-198
4 47 llama2-page-47-chunk-198


In [43]:
bm25_retriever.k=8
dense_retriever=vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 8})

In [45]:
from langchain_classic.retrievers.ensemble import EnsembleRetriever
hybrid_retriever=EnsembleRetriever(
    retrievers=[bm25_retriever, dense_retriever],
    weights=[0.5, 0.5]
)

In [46]:
hybrid_query ="Llama 2 70B grouped-query attention and inference scalability"
hybrid_docs=hybrid_retriever.invoke(hybrid_query)
display_documents(hybrid_docs,title="Hybrid Retrieval Results", max_documents=6)

Hybrid Retrieval Results
RANK: 1
PAPER PAGE: 47
SECTION: appendix
CHUNK ID: llama2-page-47-chunk-198
SOURCE: D:\GEN AI BASICS\RAG\RETRIEVER-FULL\Data\llama2-research-paper.pdf
------------------------------------------------------------------------------------------
key and value projections can be shared across multiple heads without much degradation of performance
(Chowdheryetal.,2022). Eithertheoriginalmulti-queryformatwithasingleKVprojection(MQA, Shazeer,
2019) or a grouped-query attention variant with 8 KV projections (GQA, Ainslie et al., 2023) can be used.
In Table 18, we compare MQA and GQA variants with an MHA baseline. We train all models with 150B
tokens while keeping a fixed 30B model size. To keep a similar overall parameter count across GQA an

RANK: 2
PAPER PAGE: 6
SECTION: pretraining
CHUNK ID: llama2-page-6-chunk-15
SOURCE: D:\GEN AI BASICS\RAG\RETRIEVER-FULL\Data\llama2-research-paper.pdf
--------------------------------------------------------------------------------

## QUERY DECOMPOSITION

In [61]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="openai/gpt-oss-20b:free",
    temperature=0,
    api_key=os.environ["open_router"],
    base_url="https://openrouter.ai/api/v1"
)

In [62]:
from langchain_core.prompts import ChatPromptTemplate
query_rewriting_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You rewrite conversational questions into clear,
standalone search queries.

Rules:
1. Do not answer the question.
2. Preserve important entities, dates, and technical terms.
3. Resolve pronouns using the conversation history.
4. Remove unnecessary conversational words.
5. Return only one rewritten query."""
    ),
    (
        "human",
        """Conversation history:
{chat_history}

Current query:
{query}"""
    )
])

In [63]:
query_rewritting_chain=(query_rewriting_prompt | llm | StrOutputParser())

In [64]:
chat_history = """"
User: How was Llama 2-Chat initially fine-tuned?
Assistant: It first underwent supervised fine-tuning.
"""
original_query="what did meta do after that?"
rewritten_query=query_rewritting_chain.invoke(
    {"chat_history": chat_history, "query": original_query}
).strip()
print(f"Original Query: {original_query}")
print(f"Rewritten Query: {rewritten_query}")

Original Query: what did meta do after that?
Rewritten Query: What did Meta do after supervised fine‑tuning of Llama 2‑Chat?


In [ ]:
class ex